# Level 5: EXPERT  -  Data Mining Tingkat Lanjut & Siap Produksi

**Estimasi waktu:** 6-8 jam

**Prasyarat:** Selesaikan `04_level_extra_high.ipynb` terlebih dahulu.

## Tujuan Pembelajaran

Setelah menyelesaikan level ini, Anda akan mampu:

1. Menambang **pola asosiasi** (market basket analysis) dengan algoritma **Apriori** dan membaca metrik support, confidence, lift, leverage, conviction
2. Mendeteksi **anomali** tanpa label dengan Robust Z-Score, **Isolation Forest**, dan **Local Outlier Factor**
3. Membangun model tabular modern: **HistGradientBoosting** (native categorical & missing) dan **Stacking Ensemble**
4. Mengukur performa prosedur tuning secara jujur dengan **Nested Cross-Validation**
5. Menilai dan memperbaiki **kalibrasi probabilitas** (Brier score, ECE, reliability diagram)
6. Mengambil **keputusan berbasis biaya** (cost-sensitive threshold) dan membaca **lift/gain chart**
7. Melaporkan metrik beserta **selang kepercayaan bootstrap**
8. Menginterpretasi model dengan **permutation importance** dan **Partial Dependence / ICE**
9. Mereduksi dimensi dengan **PCA**
10. Menyiapkan model untuk produksi: **persistensi model**, **monitoring data drift (PSI)**, **experiment tracking**, dan **model card**

## Peta Isi

| Bagian | Topik | Teknik Utama |
|:---|:---|:---|
| A | Association Rule Mining | Apriori, lift, leverage, conviction |
| B | Anomaly Detection | Robust Z-Score (MAD), Isolation Forest, LOF |
| C | Model Tabular Lanjutan | HistGradientBoosting, Stacking, Nested CV |
| D | Kalibrasi Probabilitas | Brier, ECE, isotonic / sigmoid calibration |
| E | Keputusan Berbasis Biaya | Bayes-optimal threshold, kurva biaya, lift/gain |
| F | Evaluasi Akhir & Ketidakpastian | Bootstrap confidence interval |
| G | Interpretasi Model | Permutation importance, PDP, ICE |
| H | Reduksi Dimensi | PCA, scree plot, PCA di dalam pipeline |
| I | Deployment & Monitoring | joblib, PSI drift, experiment tracker, model card |

> Semua fungsi pendukung berada di paket lokal **`dmtoolkit/`** (lengkap dengan unit test di folder `tests/`). Buka kodenya untuk melihat implementasi setiap rumus.

---

## Teori: Dari Model Akurat ke Keputusan yang Dapat Dipertanggungjawabkan

Level 1-4 berfokus pada pertanyaan *"seberapa akurat model saya?"*. Di dunia nyata, pertanyaan yang lebih penting adalah:

| Pertanyaan | Mengapa penting | Bagian |
|:---|:---|:---|
| Pola apa yang tersembunyi tanpa adanya label? | Banyak data bisnis tidak memiliki target | A, B, H |
| Apakah estimasi performa saya bias optimis? | Tuning pada data yang sama menimbulkan *optimism bias* | C |
| Bisakah probabilitas model dipercaya sebagai peluang? | Keputusan finansial butuh probabilitas yang jujur | D |
| Threshold berapa yang meminimalkan kerugian? | Biaya FP dan FN hampir tidak pernah sama | E |
| Seberapa yakin saya terhadap angka metrik? | Satu angka tanpa rentang ketidakpastian menyesatkan | F |
| Mengapa model memprediksi seperti itu? | Kepercayaan pengguna & audit | G |
| Apakah model masih valid bulan depan? | Data berubah (*drift*) setelah deployment | I |

### Posisi dalam CRISP-DM

```
 Business        Data            Data           Modeling       Evaluation      Deployment
 Understanding -> Understanding -> Preparation -> (C, A, B, H) -> (D, E, F, G) -> (I)
      ^                                                                            |
      +-------------------- monitoring drift memicu siklus baru <-----------------+
```

## 0. Persiapan

In [ ]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sklearn
from scipy.stats import loguniform, randint

from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (
    HistGradientBoostingClassifier, IsolationForest,
    RandomForestClassifier, StackingClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, brier_score_loss,
    precision_recall_curve, roc_auc_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV, StratifiedKFold, cross_val_predict, cross_validate, train_test_split,
)
from sklearn.neighbors import LocalOutlierFactor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, RobustScaler, StandardScaler

# Paket lokal: cari folder dmtoolkit dari direktori kerja notebook
ROOT = Path.cwd()
if not (ROOT / 'dmtoolkit').exists() and (ROOT.parent / 'dmtoolkit').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import dmtoolkit
from dmtoolkit.association import (
    apriori, association_rules, encode_transactions, format_itemset, itemset_support,
)
from dmtoolkit.datasets import make_churn_dataset, make_market_basket, make_transaction_anomalies
from dmtoolkit.drift import drift_report
from dmtoolkit.evaluation import (
    bayes_optimal_threshold, bootstrap_ci, decision_cost, expected_calibration_error,
    lift_table, nested_cross_validation, optimal_threshold, reliability_table,
)
from dmtoolkit.experiment import ExperimentTracker, ModelCard, dataframe_fingerprint, environment_info

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

RANDOM_STATE = 42
ARTIFACT_DIR = ROOT / 'artifacts'          # folder output (diabaikan oleh git)
ARTIFACT_DIR.mkdir(exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams['figure.dpi'] = 100

for k, v in environment_info().items():
    print(f'{k:11s}: {v}')
print(f'dmtoolkit  : {dmtoolkit.__version__}')
print(f'Random State: {RANDOM_STATE}')

---

## BAGIAN A: Association Rule Mining (Market Basket Analysis)

### Teori: Mencari Item yang Sering Dibeli Bersama

Diberikan $N$ transaksi, untuk aturan $A \Rightarrow C$ (*antecedent* $\Rightarrow$ *consequent*):

| Metrik | Rumus | Arti |
|:---|:---|:---|
| Support | $s(X) = \dfrac{\#\{\text{transaksi memuat } X\}}{N}$ | Seberapa sering itemset muncul |
| Confidence | $c(A \Rightarrow C) = \dfrac{s(A \cup C)}{s(A)}$ | Peluang membeli $C$ jika membeli $A$, yaitu $P(C \mid A)$ |
| Lift | $\dfrac{c(A \Rightarrow C)}{s(C)}$ | $>1$ asosiasi positif, $=1$ independen, $<1$ substitusi |
| Leverage | $s(A \cup C) - s(A)\,s(C)$ | Selisih terhadap kondisi independen (skala absolut) |
| Conviction | $\dfrac{1 - s(C)}{1 - c(A \Rightarrow C)}$ | $\infty$ jika aturan tidak pernah salah |

### Prinsip Apriori (Downward Closure)

> Jika sebuah itemset **tidak** frequent, maka semua superset-nya **pasti tidak** frequent.

```
Level 1:   {roti}  {selai}  {kopi}  {popok} ...      <- hitung support, buang yang < min_support
              \     /         |
Level 2:   {roti, selai}   {kopi, gula} ...          <- gabung (join) itemset level-1 yang lolos
                 |                                       buang (prune) kandidat yang subset-nya gagal
Level 3:   {pasta, saus tomat, keju} ...             <- ulangi sampai tidak ada kandidat
```

Tanpa prinsip ini, $d$ item menghasilkan $2^d - 1$ kandidat itemset (17 item -> 131.071 kandidat; 100 item -> $\approx 1{,}27 \times 10^{30}$).

In [ ]:
# Simulasi 3000 transaksi minimarket (data sintetis dengan pola yang ditanam)
transactions = make_market_basket(n_transactions=3000, random_state=RANDOM_STATE)
for i, t in enumerate(transactions[:5], 1):
    print(f'Transaksi {i}: {t}')

basket = encode_transactions(transactions)
print(f'\nMatriks one-hot: {basket.shape[0]} transaksi x {basket.shape[1]} item')
print(f'Rata-rata item per transaksi: {basket.sum(axis=1).mean():.2f}')

item_support = basket.mean().sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(item_support.index, item_support.values, color='steelblue', edgecolor='white')
ax.set_xlabel('Support (proporsi transaksi)')
ax.set_title('Support per Item', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
MIN_SUPPORT = 0.02
freq_itemsets = apriori(basket, min_support=MIN_SUPPORT)

print(f'Frequent itemset (min_support={MIN_SUPPORT}): {len(freq_itemsets)}')
display(freq_itemsets['length'].value_counts().sort_index().rename('jumlah').to_frame())

top_multi = freq_itemsets[freq_itemsets['length'] >= 2].head(10).copy()
top_multi['itemsets'] = top_multi['itemsets'].map(format_itemset)
display(top_multi)

In [ ]:
# Sanity check: bandingkan support Apriori dengan perhitungan brute force
rng = np.random.default_rng(RANDOM_STATE)
sample = freq_itemsets.sample(25, random_state=RANDOM_STATE)
max_diff = max(
    abs(row.support - itemset_support(basket, sorted(row.itemsets)))
    for row in sample.itertuples()
)
print(f'Selisih maksimum support Apriori vs brute force (25 itemset acak): {max_diff:.2e}')
assert max_diff < 1e-12, 'Implementasi Apriori tidak konsisten!'
print('Implementasi Apriori konsisten dengan perhitungan langsung.')

In [ ]:
rules = association_rules(freq_itemsets, metric='lift', min_threshold=1.2)
rules = rules[rules['confidence'] >= 0.30].reset_index(drop=True)

rules_view = rules.copy()
rules_view['antecedents'] = rules_view['antecedents'].map(format_itemset)
rules_view['consequents'] = rules_view['consequents'].map(format_itemset)
print(f'Aturan dengan lift >= 1.2 dan confidence >= 0.30: {len(rules)}')
display(rules_view.head(15).round(3))

In [ ]:
# Peta aturan: support vs confidence, warna = lift
fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(rules['support'], rules['confidence'], c=rules['lift'], cmap='viridis',
                s=40 + 25 * rules['lift'], alpha=0.8, edgecolors='k', linewidths=0.3)
plt.colorbar(sc, ax=ax, label='Lift')
# Beri label pada 5 pasangan item dengan leverage tertinggi (leverage simetris: A->C = C->A)
pair_key = rules.apply(lambda r: r['antecedents'] | r['consequents'], axis=1)
to_label = rules.loc[~pair_key.duplicated()].nlargest(5, 'leverage')
for j, (_, r) in enumerate(to_label.iterrows()):
    ax.annotate(f"{format_itemset(r['antecedents'])} -> {format_itemset(r['consequents'])}",
                (r['support'], r['confidence']), fontsize=8.5, xytext=(-10, 12 + 10 * (j % 2)),
                textcoords='offset points', ha='right',
                arrowprops={'arrowstyle': '-', 'color': 'gray', 'lw': 0.6})
ax.set_xlabel('Support')
ax.set_ylabel('Confidence')
ax.set_title('Peta Aturan Asosiasi', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Verifikasi: apakah pola yang ditanam di data simulasi berhasil ditemukan?
planted = [('{roti}', '{selai}'), ('{kopi}', '{gula}'), ('{popok}', '{susu bayi}'),
           ('{mie instan}', '{kerupuk}'), ('{pasta, saus tomat}', '{keju}'), ('{sampo}', '{sabun}')]
all_rules = association_rules(freq_itemsets, metric='lift', min_threshold=0)
lookup = {(format_itemset(a), format_itemset(c)): (s, conf, lift) for a, c, s, conf, lift in
          zip(all_rules['antecedents'], all_rules['consequents'], all_rules['support'],
              all_rules['confidence'], all_rules['lift'])}

check = pd.DataFrame([
    {'aturan': f'{a} -> {c}', 'ditemukan': (a, c) in lookup,
     'support': lookup.get((a, c), (np.nan,) * 3)[0],
     'confidence': lookup.get((a, c), (np.nan,) * 3)[1],
     'lift': lookup.get((a, c), (np.nan,) * 3)[2]}
    for a, c in planted
])
display(check.round(3))

In [ ]:
# Trade-off min_support: jumlah itemset dan waktu komputasi
supports = [0.20, 0.10, 0.05, 0.03, 0.02, 0.01, 0.005]
stats = []
for s in supports:
    t0 = time.perf_counter()
    res = apriori(basket, min_support=s)
    stats.append({'min_support': s, 'n_itemset': len(res),
                  'max_len': int(res['length'].max()) if len(res) else 0,
                  'detik': time.perf_counter() - t0})
stats = pd.DataFrame(stats)
display(stats.round(4))

fig, ax1 = plt.subplots(figsize=(9, 4.5))
ax1.plot(stats['min_support'], stats['n_itemset'], 'o-', color='steelblue', label='Jumlah itemset')
ax1.set_xscale('log'); ax1.set_yscale('log')
ax1.set_xlabel('min_support (skala log)'); ax1.set_ylabel('Jumlah frequent itemset', color='steelblue')
ax2 = ax1.twinx()
ax2.plot(stats['min_support'], stats['detik'], 's--', color='tomato', label='Waktu')
ax2.set_ylabel('Waktu (detik)', color='tomato')
ax1.invert_xaxis()
ax1.set_title('Semakin kecil min_support -> ledakan kombinasi', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### Interpretasi & Jebakan Umum

1. **Lift tinggi + support sangat kecil = bisa kebetulan.** Selalu lihat support (berapa transaksi nyata yang mendukung aturan).
2. **Aturan bukan kausalitas.** `{roti} -> {selai}` tidak berarti promosi roti *menyebabkan* penjualan selai naik; perlu uji A/B.
3. **Item sangat populer** (mis. air mineral) muncul sebagai consequent di banyak aturan dengan confidence tinggi tetapi lift ~1 -> tidak informatif. Itulah alasan lift lebih penting daripada confidence.
4. **Pemilihan `min_support`** adalah trade-off antara kelengkapan pola dan biaya komputasi (lihat grafik di atas).
5. **Tindakan bisnis**: penempatan rak berdekatan, bundling, rekomendasi "sering dibeli bersama".

---

## BAGIAN B: Anomaly Detection (Unsupervised)

### Teori: Jenis Anomali dan Metodenya

| Jenis | Contoh | Bisa dideteksi univariat? |
|:---|:---|:---|
| Global (point) | Transaksi Rp 4 juta saat rata-rata Rp 150 ribu | Ya |
| Kontekstual / multivariat | Rp 320 ribu (wajar) tetapi pukul 03.00, 25 km dari rumah, merchant baru | Tidak |
| Kolektif | Rangkaian transaksi kecil beruntun (*card testing*) | Perlu fitur urutan/waktu |

| Metode | Ide | Skor anomali |
|:---|:---|:---|
| Robust Z-Score | $z = 0{,}6745\,\dfrac{x - \text{median}}{\text{MAD}}$, tahan terhadap outlier | $\max_j \lvert z_j \rvert$ |
| Isolation Forest | Anomali lebih cepat **terisolasi** oleh pemotongan acak (path pendek) | rata-rata panjang path (dibalik) |
| Local Outlier Factor | Kepadatan lokal titik dibanding tetangganya | rasio kepadatan (LOF $\gg 1$ = anomali) |

> Pada kasus nyata label anomali **jarang tersedia**. Di sini data disimulasikan sehingga kita bisa mengukur performa detektor secara objektif. Karena anomali sangat jarang (~2%), gunakan **PR-AUC** dan **Precision@k**, bukan accuracy.

In [ ]:
X_tx, y_tx, kind_tx = make_transaction_anomalies(n_normal=3000, n_anomalies=60,
                                                 random_state=RANDOM_STATE, return_kind=True)
print(f'Ukuran data: {X_tx.shape}, tingkat anomali: {y_tx.mean():.2%}')
display(kind_tx.value_counts().rename('jumlah').to_frame())
display(X_tx.groupby(kind_tx).median().round(2))

In [ ]:
def robust_zscore(X):
    med = X.median()
    mad = (X - med).abs().median().replace(0, np.nan)
    z = 0.6745 * (X - med) / mad
    return z.abs().max(axis=1).fillna(0).to_numpy()

def precision_at_k(y_true, score, k):
    top_k = np.argsort(-score)[:k]
    return np.asarray(y_true)[top_k].mean()

iso = IsolationForest(n_estimators=400, contamination='auto', random_state=RANDOM_STATE)
iso.fit(X_tx)

X_tx_scaled = RobustScaler().fit_transform(X_tx)
lof = LocalOutlierFactor(n_neighbors=35)
lof.fit(X_tx_scaled)

anomaly_scores = {
    'Robust Z-Score': robust_zscore(X_tx),
    'Isolation Forest': -iso.score_samples(X_tx),
    'LOF': -lof.negative_outlier_factor_,
}

k = int(y_tx.sum())
rows = []
for name, score in anomaly_scores.items():
    top_k = np.argsort(-score)[:k]
    caught = kind_tx.iloc[top_k].value_counts()
    rows.append({
        'metode': name,
        'ROC-AUC': roc_auc_score(y_tx, score),
        'PR-AUC': average_precision_score(y_tx, score),
        f'Precision@{k}': precision_at_k(y_tx, score, k),
        'global tertangkap': f"{caught.get('global', 0)}/{(kind_tx == 'global').sum()}",
        'kontekstual tertangkap': f"{caught.get('kontekstual', 0)}/{(kind_tx == 'kontekstual').sum()}",
    })
anomaly_results = pd.DataFrame(rows).set_index('metode')
display(anomaly_results.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

for name, score in anomaly_scores.items():
    prec, rec, _ = precision_recall_curve(y_tx, score)
    axes[0].plot(rec, prec, label=f'{name} (AP={average_precision_score(y_tx, score):.2f})')
axes[0].axhline(y_tx.mean(), ls='--', color='gray', label='Acak')
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall: Detektor Anomali', fontsize=12, fontweight='bold')
axes[0].legend()

sc = axes[1].scatter(X_tx['jam'], X_tx['jarak_km'], c=anomaly_scores['Isolation Forest'],
                     cmap='magma_r', s=12, alpha=0.7)
for jenis, marker in [('global', 's'), ('kontekstual', 'X')]:
    m = kind_tx == jenis
    axes[1].scatter(X_tx.loc[m, 'jam'], X_tx.loc[m, 'jarak_km'], marker=marker, s=70,
                    facecolors='none', edgecolors='blue', linewidths=1.2, label=f'Anomali {jenis}')
plt.colorbar(sc, ax=axes[1], label='Skor Isolation Forest')
axes[1].set_xlabel('Jam transaksi'); axes[1].set_ylabel('Jarak dari rumah (km)')
axes[1].set_title('Skor Isolation Forest (jam vs jarak)', fontsize=12, fontweight='bold')
axes[1].legend(loc='upper right')
plt.tight_layout()
plt.show()

### Mengapa LOF Gagal pada Anomali Kontekstual? (Masking Effect)

Anomali kontekstual di data ini **mirip satu sama lain** (dini hari, jauh, merchant baru) sehingga membentuk *cluster kecil yang padat*. Jika `n_neighbors` lebih kecil dari ukuran cluster tersebut (40 titik), tetangga terdekat setiap anomali adalah sesama anomali -> kepadatan lokalnya terlihat "normal". Fenomena ini disebut **masking**. Mari uji sensitivitas `n_neighbors`.

In [ ]:
lof_rows = []
for n_neighbors in [10, 20, 35, 50, 80, 150]:
    lof_k = LocalOutlierFactor(n_neighbors=n_neighbors).fit(X_tx_scaled)
    score_k = -lof_k.negative_outlier_factor_
    top_k = np.argsort(-score_k)[:k]
    lof_rows.append({
        'n_neighbors': n_neighbors,
        'PR-AUC': average_precision_score(y_tx, score_k),
        f'Precision@{k}': precision_at_k(y_tx, score_k, k),
        'kontekstual tertangkap': int((kind_tx.iloc[top_k] == 'kontekstual').sum()),
    })
display(pd.DataFrame(lof_rows).round(3))

### Insight Anomaly Detection

- **Robust Z-Score** menangkap semua anomali *global* (nilai ekstrem pada satu kolom), tetapi hanya sebagian anomali *kontekstual* karena setiap kolomnya tampak wajar.
- **Isolation Forest** memotong ruang fitur secara multivariat sehingga kombinasi yang langka cepat terisolasi; pada data ini ia unggul di PR-AUC dan Precision@k.
- **LOF** sangat bergantung pada `n_neighbors`: terlalu kecil -> *masking* pada cluster anomali; terlalu besar -> kehilangan sifat "lokal". LOF juga sensitif terhadap **skala** fitur.
- Tidak ada detektor yang terbaik untuk semua jenis anomali. Dalam praktik sering dipakai **ensemble skor** (mis. rata-rata peringkat beberapa detektor).
- Parameter `contamination` hanya memengaruhi **threshold label** (`predict`), bukan urutan skor. Pilih jumlah kasus yang sanggup diperiksa tim (misalnya 50 per hari) lalu ambil top-k skor.
- Hasil detektor sebaiknya diverifikasi manusia (*human-in-the-loop*); umpan baliknya bisa menjadi label untuk model supervised di masa depan.

---

## BAGIAN C: Model Tabular Lanjutan

### Teori

**HistGradientBoosting (HGB)** adalah implementasi gradient boosting yang mengelompokkan nilai fitur ke dalam histogram (maks. 255 bin), sehingga:
- jauh lebih cepat pada data besar,
- mendukung **missing value secara native** (tidak perlu imputer),
- mendukung **fitur kategorikal secara native** (tidak perlu one-hot).

**Stacking** menggabungkan prediksi beberapa model dasar menggunakan *meta-model* yang dilatih pada prediksi **out-of-fold**:

```
             +-----------------+
 X  ------>  | Logistic Reg.   | --+
             +-----------------+   |    prediksi out-of-fold    +--------------+
 X  ------>  | Random Forest   | --+-------------------------> | Meta-model   | --> prediksi akhir
             +-----------------+   |                           | (LogReg)     |
 X  ------>  | HistGradBoost   | --+                           +--------------+
             +-----------------+
```

**Kolom ID** (`id_pelanggan`) harus dibuang: nilainya unik per baris sehingga model bisa "menghafal" dan tidak bermakna untuk pelanggan baru.

In [ ]:
churn_df = make_churn_dataset(n_samples=4000, random_state=RANDOM_STATE)
print(f'Ukuran data: {churn_df.shape} | churn rate: {churn_df["churn"].mean():.1%}')
display(churn_df.head())

quality = pd.DataFrame({
    'dtype': churn_df.dtypes.astype(str),
    'missing': churn_df.isna().sum(),
    'missing_pct': (churn_df.isna().mean() * 100).round(1),
    'unique': churn_df.nunique(),
})
display(quality)

In [ ]:
TARGET = 'churn'
ID_COLS = ['id_pelanggan']                     # bukan fitur -> dibuang

X = churn_df.drop(columns=[TARGET, *ID_COLS])
y = churn_df[TARGET]

numeric_features = X.select_dtypes(include='number').columns.tolist()
categorical_features = X.select_dtypes(exclude='number').columns.tolist()
print('Numerik    :', numeric_features)
print('Kategorikal:', categorical_features)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f'\nTrain: {X_train.shape} | Test: {X_test.shape} (test disimpan sampai Bagian F)')

In [ ]:
# --- Preprocessor untuk model linear: imputasi + scaling + one-hot ---
linear_preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')),
                      ('scaler', StandardScaler())]), numeric_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                      ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]),
     categorical_features),
])

# --- Preprocessor untuk tree: one-hot, imputasi sederhana ---
tree_preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), numeric_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                      ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]),
     categorical_features),
])

# --- Preprocessor untuk HGB: kategori -> kode ordinal, missing dibiarkan NaN ---
hgb_preprocessor = ColumnTransformer([
    ('num', 'passthrough', numeric_features),
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan,
                           encoded_missing_value=np.nan), categorical_features),
])
categorical_mask = [False] * len(numeric_features) + [True] * len(categorical_features)

def make_hgb(**params):
    return Pipeline([
        ('preprocess', hgb_preprocessor),
        ('model', HistGradientBoostingClassifier(categorical_features=categorical_mask,
                                                 random_state=RANDOM_STATE, **params)),
    ])

models = {
    'Baseline (prior)': Pipeline([('preprocess', linear_preprocessor),
                                  ('model', DummyClassifier(strategy='prior'))]),
    'Logistic Regression': Pipeline([('preprocess', linear_preprocessor),
                                     ('model', LogisticRegression(max_iter=2000))]),
    'Random Forest': Pipeline([('preprocess', tree_preprocessor),
                               ('model', RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                                                random_state=RANDOM_STATE, n_jobs=-1))]),
    'HistGradientBoosting': make_hgb(learning_rate=0.05, max_iter=200, max_leaf_nodes=12,
                                     min_samples_leaf=40),
}
models['Stacking'] = StackingClassifier(
    estimators=[('lr', models['Logistic Regression']),
                ('rf', models['Random Forest']),
                ('hgb', models['HistGradientBoosting'])],
    final_estimator=LogisticRegression(max_iter=2000),
    stack_method='predict_proba',
    cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
    n_jobs=-1,
)
print(f'{len(models)} model siap dibandingkan.')

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {'roc_auc': 'roc_auc', 'pr_auc': 'average_precision',
           'brier': 'neg_brier_score', 'bal_acc': 'balanced_accuracy'}

cv_rows, cv_raw = [], {}
for name, model in models.items():
    res = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
    cv_raw[name] = res
    cv_rows.append({
        'model': name,
        'ROC-AUC': res['test_roc_auc'].mean(), 'ROC-AUC std': res['test_roc_auc'].std(),
        'PR-AUC': res['test_pr_auc'].mean(), 'PR-AUC std': res['test_pr_auc'].std(),
        'Brier': -res['test_brier'].mean(),
        'Bal. Acc (t=0.5)': res['test_bal_acc'].mean(),
        'fit (detik)': res['fit_time'].mean(),
    })
cv_table = pd.DataFrame(cv_rows).set_index('model').sort_values('PR-AUC', ascending=False)
display(cv_table.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))
order = cv_table.index[::-1]
ax.barh(order, cv_table.loc[order, 'PR-AUC'], xerr=cv_table.loc[order, 'PR-AUC std'],
        color='steelblue', edgecolor='white', capsize=4)
ax.axvline(y_train.mean(), ls='--', color='gray', label=f'PR-AUC acak = {y_train.mean():.2f}')
ax.set_xlabel('PR-AUC (5-fold CV, rata-rata +/- std)')
ax.set_title('Perbandingan Model Churn', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Apakah selisih antar model bermakna? Bandingkan PER FOLD (fold yang sama = perbandingan berpasangan)
ref = 'Logistic Regression'
paired = pd.DataFrame({
    name: cv_raw[name]['test_pr_auc'] - cv_raw[ref]['test_pr_auc']
    for name in cv_raw if name not in (ref, 'Baseline (prior)')
})
paired.index = [f'fold {i}' for i in range(1, len(paired) + 1)]
display(paired.round(4))
summary_diff = pd.DataFrame({'selisih rata2 vs LogReg': paired.mean(),
                             'std selisih': paired.std(),
                             'fold menang': (paired > 0).sum()})
display(summary_diff.round(4))

**Cara membaca:** model yang unggul di **semua fold** dengan selisih rata-rata lebih besar dari std selisihnya memberikan bukti yang jauh lebih kuat daripada sekadar membandingkan dua angka rata-rata. Data churn ini memuat efek ambang (mis. login < 8, pemakaian < 10 GB) dan interaksi (keluhan x kontrak bulanan), sehingga model berbasis pohon diuntungkan; pada data yang hubungannya mendekati linear, Logistic Regression sering setara dan jauh lebih sederhana.

**Keputusan pemilihan model (trade-off yang disadari):** Stacking dan Random Forest sedikit unggul atas HGB *default*. Kita tetap melanjutkan dengan HGB karena (1) tuning dapat menutup sebagian selisih, (2) waktu latih HGB jauh lebih singkat daripada Stacking (lihat kolom `fit (detik)`), (3) satu model lebih mudah dipelihara, dikalibrasi ulang, dan diinterpretasi, serta (4) HGB menangani missing value dan kategori secara native. Jika selisih performa bernilai bisnis besar, Stacking layak dipilih - lihat Latihan 3.

### Nested Cross-Validation: Mengukur Prosedur Tuning secara Jujur

`best_score_` dari `GridSearchCV`/`RandomizedSearchCV` **bias optimis**: kombinasi parameter dipilih *karena* skornya tertinggi pada fold yang sama. Nested CV memisahkan dua tugas:

```
OUTER LOOP (estimasi performa)            INNER LOOP (pemilihan hyperparameter)
+--------------------------------+        +----------------------------------+
| Fold 1: [ train ......  ][test]| -----> | RandomizedSearchCV pada 'train'  |
| Fold 2: [ train ... ][test][..]|        | (cv=3) -> parameter terbaik      |
| ...                            |        +----------------------------------+
| skor outer = data yang TIDAK pernah dilihat proses tuning                  |
+----------------------------------------------------------------------------+
```

Selisih skor inner dan outer dipengaruhi **dua efek yang berlawanan**:
1. **Bias seleksi (optimis):** `best_score_` adalah nilai *maksimum* dari banyak kandidat yang berisik -> cenderung terlalu tinggi. Makin banyak kandidat (`n_iter`), makin besar biasnya.
2. **Ukuran data inner lebih kecil (pesimis):** model di inner loop dilatih pada 2/3 data outer-train, sedangkan skor outer berasal dari model yang dilatih pada seluruh outer-train.

Nilai yang dilaporkan sebagai estimasi performa adalah **skor outer**, bukan `best_score_`.

In [ ]:
param_distributions = {
    'model__learning_rate': loguniform(0.02, 0.3),
    'model__max_iter': randint(100, 400),
    'model__max_leaf_nodes': randint(8, 48),
    'model__min_samples_leaf': randint(10, 80),
    'model__l2_regularization': loguniform(1e-3, 10),
}

hgb_search = RandomizedSearchCV(
    make_hgb(), param_distributions=param_distributions, n_iter=12,
    scoring='average_precision', cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
    random_state=RANDOM_STATE, n_jobs=-1,
)

t0 = time.perf_counter()
nested = nested_cross_validation(
    hgb_search, X_train, y_train,
    outer_cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE + 1),
)
print(f'Waktu nested CV: {time.perf_counter() - t0:.1f} detik')
display(nested[['fold', 'inner_best_score', 'outer_score']].round(4))

bias = (nested['inner_best_score'] - nested['outer_score']).mean()
print(f'Rata-rata skor inner (best_score_): {nested["inner_best_score"].mean():.4f}')
print(f'Rata-rata skor outer (jujur)      : {nested["outer_score"].mean():.4f} '
      f'+/- {nested["outer_score"].std():.4f}')
print(f'Selisih inner - outer             : {bias:+.4f} '
      f'({"bias seleksi dominan" if bias > 0 else "efek ukuran data inner dominan"})')
print('\nParameter terbaik per fold (perhatikan variasinya):')
for _, r in nested.iterrows():
    print(f"  Fold {r['fold']}: " + ', '.join(f"{k.split('__')[1]}={v:.3g}" for k, v in r['best_params'].items()))

In [ ]:
# Tuning final di seluruh data train (prosedur yang performanya sudah diestimasi oleh nested CV)
hgb_search.fit(X_train, y_train)
best_hgb = hgb_search.best_estimator_
print('Parameter terpilih:')
for k, v in hgb_search.best_params_.items():
    print(f'  {k:28s}: {v:.4g}')
print(f'CV PR-AUC (inner, optimis): {hgb_search.best_score_:.4f}')

---

## BAGIAN D: Kalibrasi Probabilitas

### Teori

Model **terkalibrasi** jika dari semua pelanggan yang diberi probabilitas 0,30, sekitar 30% benar-benar churn.

| Ukuran | Rumus | Ideal |
|:---|:---|:---|
| Brier score | $\dfrac{1}{N}\sum_i (p_i - y_i)^2$ | 0 |
| Expected Calibration Error | $\sum_b \dfrac{n_b}{N}\,\lvert \bar{y}_b - \bar{p}_b \rvert$ | 0 |
| Reliability diagram | plot $\bar{p}_b$ vs $\bar{y}_b$ per bin | garis diagonal |

**Mengapa penting?** Threshold Bayes-optimal (Bagian E) dan perhitungan nilai harapan finansial **mengasumsikan** probabilitas terkalibrasi. Peringkat (ROC-AUC) yang baik tidak menjamin kalibrasi yang baik.

> `class_weight='balanced'` meningkatkan recall, tetapi **merusak kalibrasi**: probabilitas menjadi terlalu tinggi karena model "dipaksa" melihat kelas seimbang.

Agar data test tetap bersih, diagnosis di bagian ini memakai **prediksi out-of-fold (OOF)** pada data train.

In [ ]:
lr_balanced = Pipeline([('preprocess', linear_preprocessor),
                        ('model', LogisticRegression(max_iter=2000, class_weight='balanced'))])

calib_candidates = {
    'LogReg (class_weight=balanced)': lr_balanced,
    'LogReg': models['Logistic Regression'],
    'HGB tuned': best_hgb,
    'HGB tuned + sigmoid': CalibratedClassifierCV(clone(best_hgb), method='sigmoid', cv=5),
    'HGB tuned + isotonic': CalibratedClassifierCV(clone(best_hgb), method='isotonic', cv=5),
}

oof_proba = {}
for name, model in calib_candidates.items():
    oof_proba[name] = cross_val_predict(model, X_train, y_train, cv=cv, method='predict_proba')[:, 1]

calib_table = pd.DataFrame({
    name: {'ROC-AUC': roc_auc_score(y_train, p),
           'PR-AUC': average_precision_score(y_train, p),
           'Brier': brier_score_loss(y_train, p),
           'ECE': expected_calibration_error(y_train, p, n_bins=10),
           'rata2 prediksi': p.mean()}
    for name, p in oof_proba.items()
}).T
print(f'Churn rate aktual (train): {y_train.mean():.4f}')
display(calib_table.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
axes[0].plot([0, 1], [0, 1], 'k--', label='Kalibrasi sempurna')
for name in ['LogReg (class_weight=balanced)', 'HGB tuned', 'HGB tuned + isotonic']:
    tab = reliability_table(y_train, oof_proba[name], n_bins=10, strategy='quantile')
    axes[0].plot(tab['mean_predicted'], tab['fraction_positive'], 'o-',
                 label=f"{name} (ECE={expected_calibration_error(y_train, oof_proba[name]):.3f})")
axes[0].set_xlabel('Rata-rata probabilitas prediksi')
axes[0].set_ylabel('Proporsi churn aktual')
axes[0].set_title('Reliability Diagram (OOF)', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)

for name in ['LogReg (class_weight=balanced)', 'HGB tuned + isotonic']:
    axes[1].hist(oof_proba[name], bins=40, alpha=0.55, label=name)
axes[1].set_xlabel('Probabilitas churn')
axes[1].set_ylabel('Jumlah pelanggan')
axes[1].set_title('Distribusi Probabilitas', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

**Pengamatan:**
- Model `class_weight='balanced'` memiliki ROC-AUC setara, tetapi rata-rata probabilitasnya jauh di atas churn rate aktual sehingga Brier dan ECE memburuk drastis. Probabilitas seperti ini **tidak boleh** langsung dipakai untuk rumus nilai harapan.
- Kalibrasi (sigmoid/isotonic) hanya mengubah *nilai* probabilitas dengan transformasi monoton, sehingga kemampuan ranking (ROC-AUC/PR-AUC) hampir tidak berubah. Seberapa besar Brier/ECE membaik bergantung pada seberapa buruk kalibrasi awalnya; jika model sudah cukup terkalibrasi, perbaikannya kecil dan bahkan bisa sedikit memburuk akibat variansi.
- Isotonic fleksibel tetapi butuh data lebih banyak (rawan overfit pada data kecil); sigmoid (Platt) lebih stabil pada data kecil.

---

## BAGIAN E: Keputusan Berbasis Biaya

### Teori: Matriks Biaya dan Threshold Bayes-Optimal

**Asumsi ilustratif** (bukan data nyata; ganti sesuai kondisi bisnis Anda):
- Nilai pelanggan yang hilang (CLV) = Rp 600 ribu
- Biaya promo retensi per pelanggan = Rp 50 ribu
- Peluang promo berhasil mempertahankan pelanggan yang akan churn = 40%

| | Prediksi: stay (tidak diberi promo) | Prediksi: churn (diberi promo) |
|:---|:---|:---|
| **Aktual stay** | TN = 0 | FP = 50 |
| **Aktual churn** | FN = 600 | TP = 50 + (1 - 0,4) x 600 = 410 |

Beri promo jika biaya harapannya lebih kecil:

$$p \cdot C_{TP} + (1-p)\,C_{FP} \le p \cdot C_{FN} + (1-p)\,C_{TN}
\;\Longrightarrow\;
p \ge t^* = \frac{C_{FP} - C_{TN}}{(C_{FP} - C_{TN}) + (C_{FN} - C_{TP})} = \frac{50}{50 + 190} \approx 0{,}208$$

Rumus ini **hanya valid jika probabilitas terkalibrasi** - itulah kaitan Bagian D dan E.

In [ ]:
COSTS = dict(cost_fp=50.0, cost_fn=600.0, cost_tp=410.0, cost_tn=0.0)   # ribu rupiah
t_bayes = bayes_optimal_threshold(**COSTS)

p_oof = oof_proba['HGB tuned + isotonic']
t_empirical, cost_curve = optimal_threshold(y_train, p_oof, **COSTS)

def cost_per_customer(y_true, p, t):
    return decision_cost(y_true, (p >= t).astype(int), **COSTS) / len(y_true)

summary = pd.DataFrame([
    {'strategi': 'Tanpa kampanye (semua stay)', 'threshold': np.nan,
     'biaya/pelanggan': cost_per_customer(y_train, p_oof, 1.01)},
    {'strategi': 'Promo ke semua pelanggan', 'threshold': 0.0,
     'biaya/pelanggan': cost_per_customer(y_train, p_oof, 0.0)},
    {'strategi': 'Model, threshold default', 'threshold': 0.5,
     'biaya/pelanggan': cost_per_customer(y_train, p_oof, 0.5)},
    {'strategi': 'Model, threshold Bayes (teori)', 'threshold': t_bayes,
     'biaya/pelanggan': cost_per_customer(y_train, p_oof, t_bayes)},
    {'strategi': 'Model, threshold empiris (OOF)', 'threshold': t_empirical,
     'biaya/pelanggan': cost_per_customer(y_train, p_oof, t_empirical)},
])
baseline_cost = summary.loc[0, 'biaya/pelanggan']
summary['hemat vs tanpa kampanye (%)'] = (1 - summary['biaya/pelanggan'] / baseline_cost) * 100
display(summary.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(cost_curve['threshold'], cost_curve['cost_per_sample'], color='steelblue')
for t, label, color in [(0.5, 'Default 0.5', 'gray'), (t_bayes, f'Bayes {t_bayes:.3f}', 'green'),
                        (t_empirical, f'Empiris {t_empirical:.2f}', 'red')]:
    axes[0].axvline(t, ls='--', color=color, label=label)
axes[0].set_xlabel('Threshold'); axes[0].set_ylabel('Biaya per pelanggan (ribu Rp)')
axes[0].set_title('Kurva Biaya vs Threshold (OOF)', fontsize=12, fontweight='bold')
axes[0].legend()

lift = lift_table(y_train, p_oof, n_bins=10)
pct = np.r_[0, np.arange(1, 11) * 10]
axes[1].plot(pct, np.r_[0, lift['cumulative_gain']] * 100, 'o-', label='Model')
axes[1].plot([0, 100], [0, 100], 'k--', label='Acak')
axes[1].set_xlabel('% pelanggan yang disasar (urut skor tertinggi)')
axes[1].set_ylabel('% churner tertangkap')
axes[1].set_title('Cumulative Gain Chart', fontsize=12, fontweight='bold')
axes[1].legend()
plt.tight_layout()
plt.show()

display(lift[['kelompok', 'n', 'positif', 'tingkat_positif', 'cumulative_gain', 'lift', 'cumulative_lift']].round(3))
top2 = lift.loc[1, 'cumulative_gain']
print(f'Menyasar 20% pelanggan teratas menangkap {top2:.0%} churner '
      f'(lift kumulatif {lift.loc[1, "cumulative_lift"]:.2f}x dibanding acak).')

---

## BAGIAN F: Evaluasi Akhir pada Data Test & Ketidakpastian

Model final = HGB tuned + kalibrasi isotonic, dilatih pada **seluruh data train**. Threshold sudah ditetapkan **sebelum** melihat data test.

### Bootstrap Confidence Interval

Data test di-*resample* dengan pengembalian $B$ kali, metrik dihitung ulang, lalu diambil persentil 2,5% dan 97,5% sebagai CI 95%. Jika CI dua model saling tumpang tindih lebar, klaim "model A lebih baik" perlu bukti tambahan.

In [ ]:
final_model = CalibratedClassifierCV(clone(best_hgb), method='isotonic', cv=5)
final_model.fit(X_train, y_train)
DECISION_THRESHOLD = float(t_empirical)

p_test = final_model.predict_proba(X_test)[:, 1]
yhat_test = (p_test >= DECISION_THRESHOLD).astype(int)

final_metrics = {
    'roc_auc': roc_auc_score(y_test, p_test),
    'pr_auc': average_precision_score(y_test, p_test),
    'brier': brier_score_loss(y_test, p_test),
    'ece': expected_calibration_error(y_test, p_test),
    'balanced_accuracy': balanced_accuracy_score(y_test, yhat_test),
    'cost_per_customer': cost_per_customer(y_test, p_test, DECISION_THRESHOLD),
    'cost_no_campaign': cost_per_customer(y_test, p_test, 1.01),
}
print('=' * 60)
print(f'EVALUASI AKHIR - DATA TEST (n={len(y_test)}), threshold={DECISION_THRESHOLD:.2f}')
print('=' * 60)
for k, v in final_metrics.items():
    print(f'{k:20s}: {v:.4f}')

In [ ]:
B = 1000
ci = {
    'ROC-AUC': bootstrap_ci(y_test, p_test, roc_auc_score, n_resamples=B, random_state=RANDOM_STATE),
    'PR-AUC': bootstrap_ci(y_test, p_test, average_precision_score, n_resamples=B, random_state=RANDOM_STATE),
    'Biaya/pelanggan': bootstrap_ci(
        y_test, p_test, lambda yt, p: cost_per_customer(yt, p, DECISION_THRESHOLD),
        n_resamples=B, random_state=RANDOM_STATE),
    'Penghematan/pelanggan': bootstrap_ci(
        y_test, p_test,
        lambda yt, p: cost_per_customer(yt, p, 1.01) - cost_per_customer(yt, p, DECISION_THRESHOLD),
        n_resamples=B, random_state=RANDOM_STATE),
}
ci_table = pd.DataFrame({k: {'estimasi': r.estimate, 'CI 2.5%': r.lower, 'CI 97.5%': r.upper, 'SE': r.std}
                         for k, r in ci.items()}).T
display(ci_table.round(4))

saving = ci['Penghematan/pelanggan']
print(f'Penghematan per pelanggan: Rp {saving.estimate:.1f} ribu '
      f'(CI 95%: {saving.lower:.1f} - {saving.upper:.1f} ribu).')
print('Jika batas bawah CI > 0, model memberi manfaat finansial dengan keyakinan memadai '
      '(di bawah asumsi biaya yang dipakai).')

---

## BAGIAN G: Interpretasi Model

| Teknik | Pertanyaan yang dijawab | Catatan |
|:---|:---|:---|
| Permutation importance | Seberapa besar skor turun jika satu kolom diacak? | Dihitung pada data test; fitur berkorelasi bisa "berbagi" importance |
| Partial Dependence (PDP) | Bagaimana rata-rata prediksi berubah terhadap satu fitur? | Mengasumsikan fitur independen |
| ICE | Bagaimana prediksi **tiap individu** berubah? | Garis yang tidak sejajar = ada interaksi |

Karena data disimulasikan, kita bisa membandingkan hasil interpretasi dengan **pola yang sebenarnya**: risiko sangat tinggi pada 6 bulan pertama lalu turun dan melandai setelah ~24 bulan (`tenure_bulan`), melonjak bila `login_30hr` < 8 atau `penggunaan_gb` < 10, naik seiring `jumlah_keluhan_3bln` (jauh lebih tajam pada kontrak bulanan - terlihat sebagai ICE yang tidak sejajar), dan kontrak `bulanan` paling berisiko.

In [ ]:
perm = permutation_importance(final_model, X_test, y_test, scoring='average_precision',
                              n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.DataFrame({'fitur': X_test.columns, 'importance': perm.importances_mean,
                    'std': perm.importances_std}).sort_values('importance')

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(imp['fitur'], imp['importance'], xerr=imp['std'], color='steelblue', capsize=3)
ax.set_xlabel('Penurunan PR-AUC saat kolom diacak')
ax.set_title('Permutation Importance (kolom asli, data test)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# PDP butuh kolom numerik bertipe float (kolom integer bisa terbulatkan secara implisit).
# Baris dengan NaN dibuang agar grid persentil tiap fitur tidak bernilai NaN.
X_pdp = X_test.astype({c: float for c in numeric_features}).dropna()

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
PartialDependenceDisplay.from_estimator(
    final_model, X_pdp,
    features=['tenure_bulan', 'jumlah_keluhan_3bln', 'login_30hr', 'penggunaan_gb'],
    kind='both', subsample=150, percentiles=(0.01, 0.99), random_state=RANDOM_STATE, ax=axes.ravel(),
    ice_lines_kw={'alpha': 0.15, 'color': 'steelblue'}, pd_line_kw={'color': 'red', 'lw': 2.5},
)
for ax in axes.ravel():
    legend = ax.get_legend()
    if legend is not None:
        legend.remove()
fig.suptitle('Partial Dependence (merah) + ICE (biru)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

contract_effect = (pd.DataFrame({'kontrak': X_test['kontrak'], 'p': p_test, 'aktual': y_test})
                   .groupby('kontrak')[['p', 'aktual']].mean().round(3))
display(contract_effect.rename(columns={'p': 'rata2 prediksi', 'aktual': 'churn aktual'}))

---

## BAGIAN H: Reduksi Dimensi dengan PCA

### Teori

PCA mencari arah (komponen) ortogonal yang memaksimalkan varians. Untuk data terstandardisasi $Z$, komponen adalah eigenvektor matriks kovarians $\frac{1}{n-1} Z^\top Z$ dan *explained variance ratio* komponen ke-$k$ adalah $\lambda_k / \sum_j \lambda_j$.

Kegunaan: visualisasi data berdimensi tinggi, mengurangi multikolinearitas, mempercepat model. Kekurangan: komponen sulit diinterpretasi dan PCA tidak memakai informasi target.

> PCA **harus** di-fit hanya pada data train (masukkan ke dalam Pipeline) agar tidak terjadi leakage.

In [ ]:
cancer = load_breast_cancer(as_frame=True)
X_bc, y_bc = cancer.data, cancer.target
print(f'Breast cancer: {X_bc.shape[0]} sampel x {X_bc.shape[1]} fitur')

pca_full = Pipeline([('scaler', StandardScaler()), ('pca', PCA(random_state=RANDOM_STATE))]).fit(X_bc)
evr = pca_full.named_steps['pca'].explained_variance_ratio_
cum = np.cumsum(evr)
n95 = int(np.searchsorted(cum, 0.95) + 1)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].bar(range(1, len(evr) + 1), evr, color='steelblue', label='Per komponen')
axes[0].plot(range(1, len(evr) + 1), cum, 'o-', color='darkorange', label='Kumulatif')
axes[0].axhline(0.95, ls='--', color='red', alpha=0.6)
axes[0].axvline(n95, ls='--', color='red', alpha=0.6, label=f'{n95} komponen = 95% varians')
axes[0].set_xlabel('Komponen'); axes[0].set_ylabel('Explained variance ratio')
axes[0].set_title('Scree Plot', fontsize=12, fontweight='bold'); axes[0].legend()

Z = pca_full.transform(X_bc)
for label, name, color in [(0, 'malignant', 'tomato'), (1, 'benign', 'steelblue')]:
    m = y_bc == label
    axes[1].scatter(Z[m, 0], Z[m, 1], s=14, alpha=0.6, color=color, label=name)
axes[1].set_xlabel(f'PC1 ({evr[0]:.0%})'); axes[1].set_ylabel(f'PC2 ({evr[1]:.0%})')
axes[1].set_title('Proyeksi 2D (tanpa label saat fitting)', fontsize=12, fontweight='bold')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
cv_bc = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
pca_results = []
for n_comp in [2, 5, n95, None]:
    steps = [('scaler', StandardScaler())]
    if n_comp is not None:
        steps.append(('pca', PCA(n_components=n_comp, random_state=RANDOM_STATE)))
    steps.append(('model', LogisticRegression(max_iter=5000)))
    scores = cross_validate(Pipeline(steps), X_bc, y_bc, cv=cv_bc, scoring='roc_auc')
    pca_results.append({'konfigurasi': f'PCA {n_comp} komponen' if n_comp else 'Tanpa PCA (30 fitur)',
                        'ROC-AUC': scores['test_score'].mean(), 'std': scores['test_score'].std()})
display(pd.DataFrame(pca_results).round(4))

---

## BAGIAN I: Deployment & Monitoring

### Teori: Siklus Hidup Model

```
 Train --> Simpan (joblib) --> Serve --> Monitor input (drift) & output (performa) --> Retrain?
                                              |
                     PSI < 0,10 : stabil      |   PSI 0,10 - 0,25 : pantau   |  PSI >= 0,25 : investigasi
```

**Population Stability Index (PSI)**: $\text{PSI} = \sum_i (a_i - e_i)\ln\frac{a_i}{e_i}$, dengan $e_i$ proporsi data referensi (train) dan $a_i$ proporsi data baru pada bin $i$. Ambang 0,10 / 0,25 adalah *rule of thumb* praktik industri (credit scoring), bukan standar formal.

> **Keamanan:** file `joblib`/`pickle` dapat mengeksekusi kode saat dimuat. Hanya muat model dari sumber tepercaya.

In [ ]:
model_path = ARTIFACT_DIR / 'churn_model.joblib'
joblib.dump(final_model, model_path)
reloaded = joblib.load(model_path)

same = np.allclose(reloaded.predict_proba(X_test)[:, 1], p_test)
print(f'Model disimpan: {model_path.relative_to(ROOT)} ({model_path.stat().st_size / 1024:.0f} KB)')
print(f'Prediksi model hasil load identik: {same}')
assert same

In [ ]:
# Simulasi data produksi 3 periode: stabil -> mulai berubah -> berubah besar
batches = {
    'Bulan 1 (shift=0.0)': make_churn_dataset(1500, random_state=101, shift=0.0),
    'Bulan 2 (shift=0.5)': make_churn_dataset(1500, random_state=102, shift=0.5),
    'Bulan 3 (shift=1.0)': make_churn_dataset(1500, random_state=103, shift=1.0),
}

monitor_rows, reports = [], {}
for name, batch in batches.items():
    Xb, yb = batch.drop(columns=[TARGET, *ID_COLS]), batch[TARGET]
    rep = drift_report(X_train, Xb)
    reports[name] = rep
    pb = final_model.predict_proba(Xb)[:, 1]
    monitor_rows.append({
        'periode': name,
        'kolom drift signifikan': int((rep['status'] == 'drift signifikan').sum()),
        'kolom perlu dipantau': int((rep['status'] == 'perlu dipantau').sum()),
        'PSI maks': rep['psi'].max(),
        'churn aktual': yb.mean(),
        'rata2 prediksi': pb.mean(),
        'ROC-AUC': roc_auc_score(yb, pb),
        'PR-AUC': average_precision_score(yb, pb),
        'ECE': expected_calibration_error(yb, pb),
        'biaya/pelanggan': cost_per_customer(yb, pb, DECISION_THRESHOLD),
        'biaya tanpa kampanye': cost_per_customer(yb, pb, 1.01),
    })
monitor = pd.DataFrame(monitor_rows).set_index('periode')
display(monitor.round(3))
print('\nDetail drift Bulan 3:')
display(reports['Bulan 3 (shift=1.0)'].round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
psi_matrix = pd.DataFrame({name: rep.set_index('kolom')['psi'] for name, rep in reports.items()})
psi_matrix = psi_matrix.sort_values(psi_matrix.columns[-1], ascending=False)
sns.heatmap(psi_matrix, annot=True, fmt='.3f', cmap='YlOrRd', vmin=0, vmax=0.5,
            cbar_kws={'label': 'PSI'}, ax=ax)
ax.set_title('PSI per Fitur per Periode', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretasi monitoring:**
- Bulan 1 stabil: PSI kecil di semua kolom, performa setara data test.
- Bulan 2-3: beberapa fitur bergeser (lihat heatmap). Churn aktual naik dan rata-rata prediksi ikut naik karena pergeseran terjadi pada **input** (*covariate shift*) sementara hubungan fitur-target tetap. Bandingkan `rata2 prediksi` dengan `churn aktual`: selisih yang membesar menandakan kalibrasi mulai bergeser.
- **Hati-hati membaca PR-AUC antar periode:** PR-AUC bergantung pada prevalensi kelas positif. Kenaikan PR-AUC saat churn rate naik **bukan** bukti model membaik. Untuk perbandingan antar periode, ROC-AUC dan metrik kalibrasi lebih adil.
- Biaya per pelanggan naik karena jumlah churner naik, bukan semata karena model memburuk; bandingkan selalu dengan baseline "tanpa kampanye" pada periode yang sama.
- Drift **tidak otomatis** berarti model rusak. Keputusan retrain sebaiknya berdasar kombinasi: PSI, performa pada label terbaru (yang biasanya datang terlambat), dan dampak bisnis.

In [ ]:
tracker = ExperimentTracker(ARTIFACT_DIR / 'experiments.jsonl')
tracker.clear()                                   # mulai bersih setiap notebook dijalankan ulang
fingerprint = dataframe_fingerprint(X_train.assign(churn=y_train))

for name, row in cv_table.iterrows():
    tracker.log_run(name, params={'jenis': 'baseline-cv'},
                    metrics={'cv_pr_auc': row['PR-AUC'], 'cv_roc_auc': row['ROC-AUC'], 'cv_brier': row['Brier']},
                    tags={'tahap': 'perbandingan'}, data_fingerprint=fingerprint)

tracker.log_run('HGB tuned + isotonic (final)',
                params={**hgb_search.best_params_, 'calibration': 'isotonic',
                        'threshold': DECISION_THRESHOLD},
                metrics={'cv_pr_auc': float(average_precision_score(y_train, p_oof)),
                         'nested_outer_pr_auc': float(nested['outer_score'].mean()),
                         'test_pr_auc': final_metrics['pr_auc'],
                         'test_cost_per_customer': final_metrics['cost_per_customer']},
                tags={'tahap': 'final'}, data_fingerprint=fingerprint)

runs = tracker.to_frame()
display(runs[['name', 'metrics.cv_pr_auc', 'metrics.cv_roc_auc', 'metrics.test_pr_auc', 'tags.tahap']].round(4))
print('Run terbaik (cv_pr_auc):', tracker.best_run('cv_pr_auc')['name'])
print('Sidik jari data train :', fingerprint[:16], '...')

In [ ]:
card = ModelCard(
    model_name='churn-hgb-isotonic',
    version='1.0.0',
    problem='Prediksi churn pelanggan layanan berlangganan dalam periode berikutnya (klasifikasi biner).',
    intended_use='Memprioritaskan pelanggan untuk program retensi. Bukan untuk menolak layanan atau menetapkan harga.',
    training_data={'sumber': 'make_churn_dataset (SINTETIS)', 'n_train': len(X_train),
                   'churn_rate': round(float(y_train.mean()), 4), 'fingerprint': fingerprint[:16]},
    metrics={'nested_cv_pr_auc': f"{nested['outer_score'].mean():.3f} +/- {nested['outer_score'].std():.3f}",
             'test_pr_auc': f"{ci['PR-AUC'].estimate:.3f} (CI95 {ci['PR-AUC'].lower:.3f}-{ci['PR-AUC'].upper:.3f})",
             'test_roc_auc': f"{ci['ROC-AUC'].estimate:.3f} (CI95 {ci['ROC-AUC'].lower:.3f}-{ci['ROC-AUC'].upper:.3f})",
             'test_brier': round(final_metrics['brier'], 4),
             'test_ece': round(final_metrics['ece'], 4)},
    decision_threshold=DECISION_THRESHOLD,
    limitations=[
        'Data disimulasikan; wajib divalidasi ulang pada data pelanggan nyata.',
        'Matriks biaya berupa asumsi ilustratif; threshold harus dihitung ulang jika biaya berubah.',
        'Hubungan fitur-churn bersifat asosiatif, bukan kausal.',
        'Pantau PSI bulanan; PSI >= 0.25 pada fitur utama memicu evaluasi ulang.',
    ],
    ethical_considerations=[
        'Jangan gunakan atribut sensitif (mis. region sebagai proksi demografi) untuk perlakuan yang merugikan.',
        'Evaluasi performa per segmen (region, plan) sebelum dipakai luas.',
    ],
)
card.save_json(ARTIFACT_DIR / 'model_card.json')
(ARTIFACT_DIR / 'model_card.md').write_text(card.to_markdown(), encoding='utf-8')
print(card.to_markdown())

---

## Checklist Level Expert

### Pola Tanpa Label
- [ ] Aturan asosiasi dinilai dengan **lift** dan **support**, bukan confidence saja
- [ ] Detektor anomali dievaluasi dengan PR-AUC / Precision@k (jika ada label validasi)

### Evaluasi yang Jujur
- [ ] Kolom ID dan kolom pasca-kejadian dibuang dari fitur
- [ ] Performa prosedur tuning diestimasi dengan **nested CV**
- [ ] Kalibrasi dicek (Brier, ECE, reliability diagram) sebelum probabilitas dipakai untuk keputusan
- [ ] Threshold diturunkan dari **matriks biaya** dan ditetapkan **sebelum** melihat data test
- [ ] Metrik akhir dilaporkan dengan **confidence interval**

### Siap Produksi
- [ ] Model disimpan dan diuji ulang setelah dimuat
- [ ] Rencana monitoring drift (PSI) dan performa tersedia
- [ ] Eksperimen tercatat (parameter, metrik, sidik jari data, versi library)
- [ ] Model card berisi tujuan, limitasi, dan pertimbangan etis

---

## Latihan dan Quiz

### Quiz Teori

1. Aturan `{air mineral} -> {roti}` memiliki confidence 0,35 dan lift 1,02. Apakah aturan ini berguna? Jelaskan.
2. Mengapa prinsip *downward closure* membuat Apriori jauh lebih efisien dibanding enumerasi semua kombinasi?
3. Mengapa Robust Z-Score gagal mendeteksi anomali kontekstual, sedangkan Isolation Forest berhasil?
4. Apa yang diukur oleh nested CV yang **tidak** diukur oleh `best_score_` pada `RandomizedSearchCV`?
5. Model A memiliki ROC-AUC lebih tinggi dari model B, tetapi Brier score-nya lebih buruk. Bagaimana mungkin?
6. Turunkan threshold Bayes-optimal jika $C_{FP}=20$, $C_{FN}=500$, $C_{TP}=300$, $C_{TN}=0$.
7. CI 95% PR-AUC model A = [0,61; 0,72] dan model B = [0,64; 0,75]. Apa kesimpulan yang tepat?
8. PSI fitur `biaya_bulanan` = 0,31, tetapi PR-AUC bulan ini tidak turun. Apa tindakan yang tepat?

### Kunci Jawaban Quiz

<details>
<summary>Klik untuk melihat pembahasan</summary>

1. Kurang berguna. Lift ~1 berarti pembelian air mineral hampir independen terhadap roti; confidence 0,35 hanya mencerminkan popularitas roti itu sendiri.
2. Kandidat berukuran $k$ hanya dibentuk dari itemset frequent berukuran $k-1$, dan kandidat yang punya subset tidak frequent langsung dibuang tanpa menghitung support-nya. Ruang pencarian terpangkas drastis.
3. Robust Z-Score menilai tiap kolom terpisah; anomali kontekstual memiliki nilai wajar di setiap kolom. Isolation Forest memotong ruang fitur secara multivariat sehingga kombinasi langka cepat terisolasi.
4. Nested CV mengukur performa **seluruh prosedur** (termasuk pemilihan hyperparameter) pada data yang tidak pernah dipakai untuk memilih parameter. `best_score_` bias optimis karena merupakan maksimum dari banyak skor yang berisik.
5. ROC-AUC hanya menilai **urutan** (ranking); Brier menilai **ketepatan nilai** probabilitas. Model bisa mengurutkan dengan baik tetapi probabilitasnya terlalu ekstrem atau bergeser (tidak terkalibrasi).
6. $t^* = \dfrac{20 - 0}{(20 - 0) + (500 - 300)} = \dfrac{20}{220} \approx 0{,}091$.
7. Interval tumpang tindih lebar, sehingga belum ada bukti kuat B lebih baik. Gunakan uji berpasangan (bootstrap selisih pada sampel yang sama) atau data lebih banyak.
8. Investigasi penyebab drift (perubahan harga? bug pipeline data?), pantau lebih ketat, dan periksa performa per segmen serta kalibrasi (rata-rata prediksi vs aktual). Retrain belum tentu wajib jika performa dan kalibrasi tetap baik.

</details>

In [ ]:
# LATIHAN 1: Association rules dengan batasan bisnis
# ──────────────────────────────────────────────────
# Tugas:
#  a) Jalankan apriori dengan min_support=0.01 dan max_len=3
#  b) Ambil aturan dengan lift >= 1.5 yang consequent-nya hanya 1 item
#  c) Buang aturan yang antecedent-nya memuat 'air mineral'
#  d) Urutkan berdasarkan leverage, tampilkan 10 teratas
#  e) Usulkan 2 aksi bisnis (bundling / tata letak rak) beserta alasannya
#
# Tulis kode Anda di bawah:

In [ ]:
# LATIHAN 2: Sensitivitas keputusan terhadap asumsi biaya
# ──────────────────────────────────────────────────────
# Tugas:
#  a) Variasikan peluang keberhasilan promo: 0.2, 0.4, 0.6, 0.8
#     (cost_tp = 50 + (1 - sukses) * 600)
#  b) Untuk setiap nilai, hitung threshold Bayes dan threshold empiris (gunakan p_oof)
#  c) Hitung penghematan biaya per pelanggan pada data train (OOF)
#  d) Pada nilai sukses berapa kampanye retensi TIDAK lagi menguntungkan? Mengapa?
#
# Tulis kode Anda di bawah:

In [ ]:
# LATIHAN 3: Strategi retrain saat drift
# ─────────────────────────────────────
# Tugas:
#  a) Gunakan batch 'Bulan 3 (shift=1.0)' sebagai data terbaru berlabel
#  b) Bandingkan 3 strategi pada batch baru lain: make_churn_dataset(1500, random_state=104, shift=1.0)
#       (1) model lama tanpa perubahan
#       (2) model lama + kalibrasi ulang (CalibratedClassifierCV dengan cv='prefit' atau
#           FrozenEstimator pada scikit-learn >= 1.6) menggunakan data Bulan 3
#       (3) retrain penuh dengan gabungan X_train + data Bulan 3
#  c) Laporkan PR-AUC, ECE, dan biaya per pelanggan untuk setiap strategi
#  d) Strategi mana yang Anda rekomendasikan dan mengapa?
#  e) BONUS: ulangi Bagian D-F menggunakan models['Stacking'] sebagai model dasar.
#     Apakah selisih PR-AUC-nya sepadan dengan tambahan waktu latih dan kompleksitas?
#
# Tulis kode Anda di bawah:

---

## Selamat! Anda Telah Menyelesaikan Level Expert

| Level | Yang Dipelajari |
|:------|:----------------|
| Light | EDA, kualitas data, visualisasi |
| Medium | KNN, Decision Tree, Logistic Regression, pipeline, cross-validation |
| High | GridSearchCV, class imbalance, ROC/PR, Random Forest, feature importance |
| Extra High | ColumnTransformer, pipeline end-to-end, clustering, threshold, laporan |
| **Expert** | **Apriori, anomaly detection, HGB & stacking, nested CV, kalibrasi, keputusan berbasis biaya, bootstrap CI, PDP/ICE, PCA, drift monitoring, model card** |

### Langkah Selanjutnya

1. Terapkan alur Level 5 pada dataset nyata (mis. transaksi ritel untuk Apriori, data churn telekomunikasi publik untuk klasifikasi).
2. Pelajari **SHAP** untuk interpretasi lokal per prediksi.
3. Pelajari **validasi berbasis waktu** (`TimeSeriesSplit`) bila data memiliki urutan waktu.
4. Bungkus model menjadi API (FastAPI) dan otomatisasi monitoring PSI secara terjadwal.
5. Pelajari **fairness assessment** untuk mengevaluasi dampak model terhadap kelompok berbeda.

> **Model terbaik bukan yang skornya paling tinggi, melainkan yang keputusannya paling dapat dipertanggungjawabkan: evaluasinya jujur, probabilitasnya terkalibrasi, biayanya terukur, ketidakpastiannya dilaporkan, dan perilakunya dipantau.**